In [40]:
import os
import json  # Import the json module
from pathlib import Path

from PyDI.io import load_xml, load_parquet, load_csv
from PyDI.profiling import DataProfiler

# ------------------------------------------------------------------
# Global
# ------------------------------------------------------------------
OUTPUT_DIR = "output/"

# ------------------------------------------------------------------
# Data Profiling Function
# ------------------------------------------------------------------

def profile_data(**datasets: dict[str,str]):

    all_profiles = {}
    
    for name, path in datasets.items():

        extension = os.path.splitext(path)[1]
        match extension:
            case ".parquet":
                df = load_parquet(
                    path,
                    name=name
                )
            case ".csv":
                df = load_csv(
                    path,
                    name=name
                )
            case ".xml":
                df = load_xml(
                    path,
                    name=name,
                    nested_handling="aggregate"
                )
            # TODO: Add further supported loaders from https://github.com/wbsg-uni-mannheim/PyDI/blob/main/PyDI/io/loaders.py
            
        profiler = DataProfiler()
        profile = profiler.summary(df)

        # TODO: Coverage Analysis maybe after schema matching?

        all_profiles[name] = profile
    
        # write to output
        output_path = Path(OUTPUT_DIR) / Path("profile/") / (name + "_profile_output.json")
        with open(output_path, "w") as f:
                json.dump(profile, f, indent=2)
    
    return all_profiles

# ------------------------------------------------------------------
# Define Tools
# ------------------------------------------------------------------

profile_tool = Tool(
    name="profile_data",
    func=lambda tool_input: profile_data(**json.loads(tool_input)),
    description="Profiles a dataset and returns column-wise statistics."
)


# ------------------------------------------------------------------
# Pipeline
# ------------------------------------------------------------------
def run_pipeline(datasets: dict[str,str]):

    # Step 1: Data profiling
    all_profiles = profile_tool.run(json.dumps(datasets))
    return all_profiles


# ------------------------------------------------------------------
# Usage
# ------------------------------------------------------------------
if __name__ == "__main__":
    datasets = {
        "kaggle380k": "../restaurant-integration/datasets/kaggle380k.parquet",
        "uber_eats": "../restaurant-integration/datasets/uber-eats-restaurants.csv",
    }
    
    output = run_pipeline(datasets)

kaggle380k:
  Rows: 380,358
  Columns: 20
  Total nulls: 853,673
  Null percentage: 11.2%
  Null counts per column:
    website: 94,923 (25.0%)
    map_url: 35,149 (9.2%)
    phone_raw: 48,860 (12.8%)
    phone_e164: 52,341 (13.8%)
    address_line1: 17,684 (4.6%)
    address_line2: 331,013 (87.0%)
    street: 34,160 (9.0%)
    house_number: 36,100 (9.5%)
    city: 17,712 (4.7%)
    state: 69,929 (18.4%)
    postal_code: 72,884 (19.2%)
    latitude: 218 (0.1%)
    longitude: 218 (0.1%)
    rating: 42,482 (11.2%)

uber_eats:
  Rows: 63,469
  Columns: 11
  Total nulls: 68,006
  Null percentage: 9.7%
  Null counts per column:
    score: 28,167 (44.4%)
    ratings: 28,167 (44.4%)
    category: 85 (0.1%)
    price_range: 10,617 (16.7%)
    full_address: 453 (0.7%)
    zip_code: 517 (0.8%)

